# 회의록 만들기

녹음 또는 녹취록 → 회의록 → 드라이브/노션. **위에서 아래로 순서대로** 실행합니다.

| # | 셀 | 하는 일 | 만드는 것 |
|---|---|---|---|
| 1 | 설정 | 입력 파일·제목·날짜 지정 | `AUDIO`/`TRANSCRIPT`, `CFG` |
| 2 | STT | 오디오→텍스트 (녹취록이면 건너뜀) | `transcript_path` |
| 3 | 녹취록 확인 | 어느 파일인지·내용 멀쩡한지 확인 | `transcript` |
| 4 | 추출 | `claude -p` 로 회의록 구조 추출 + 검증 | `minutes`, `bundle` |
| 5 | 검토 | 번호 붙여 보여주고 **멈춤** | — |
| 6 | 확정 | 고른 것만 md/html/json 생성 | `out` |
| 7 | 드라이브 | 점검 후 전송 | — |
| 8 | 노션 | 커넥터로 페이지 생성 | — |

**API 키는 쓰지 않습니다.** 추출은 `claude -p`(Claude Code CLI)가 담당합니다.

> ⚠ 이 노트북의 **출력에는 실제 회의 내용이 남습니다.** 커밋 전에
> `Kernel → Restart & Clear All Outputs` 를 실행하세요.

> 커널을 재시작하면 변수가 사라집니다. `out` 이 없다는 오류가 나면 6번까지 다시 실행하세요.
> 4번(추출)은 `data/minutes/draft/*.cli.json` 이 있으면 다시 돌리지 않아도 됩니다.

## 1. 설정

**하는 일** — `.env` 를 읽어 이번 실행에 쓸 값을 정합니다.

### 설정은 `.env` 한 곳에 있습니다

입력 파일·제목·날짜·STT 모델·저장 경로·전송 방식까지 **전부 [`.env`](.env) 에서** 바꿉니다.
각 항목에 설명 주석이 달려 있습니다. 없으면 `.env.example` 을 복사해서 만드세요.

```
1 입력     INPUT_AUDIO / INPUT_TRANSCRIPT / MEETING_TITLE / MEETING_DATE
2 STT      WHISPER_MODEL / WHISPER_DEVICE / STT_LANGUAGE
3 추출     EXTRACT_MODE (cli|api) / ANTHROPIC_API_KEY
4 검토     ACCEPT
5 산출물   OUTPUT_DIR / OUTPUT_LAYOUT / OUTPUT_OVERWRITE
6 전송     SEND (api|sync|manual) / DRIVE_* / SYNC_DIR
7 노션     NOTION_TARGET
```

> **`.env` 를 고쳤으면 커널을 재시작**해야 반영됩니다 (`load_dotenv` 는 import 때 1회만 실행).

### 이 셀에서 임시로 덮어쓰기

아래 `OVERRIDE` 는 «이번 한 번만» 다르게 하고 싶을 때 씁니다. 비워두면 `.env` 값을 씁니다.
상시 설정은 `.env` 에 쓰세요 — 노트북에 적으면 다음에 또 고쳐야 합니다.

In [2]:
# ─── 이번 실행만 다르게 하고 싶을 때. 비우면 .env 값 사용 ───
OVERRIDE = {
    # 'input_transcript': r'data/transcripts/xxx.txt',
    # 'input_audio':      r'data/audio/xxx.m4a',
    # 'meeting_title':    '킥오프',
    # 'meeting_date':     '2026-08-26',
}

# ─── 환경 준비 (여기부터는 건드릴 것 없음) ───
import importlib, os, sys
from pathlib import Path

os.environ.setdefault('PYTHONIOENCODING', 'utf-8')
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    raise SystemExit(f'meeting_minutes 폴더에서 열어야 합니다. 현재: {ROOT}')
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

#  config.py 는 autoreload 대상에서 «제외» 한다.
#  Config 가 dataclass 라서 필드가 바뀌면 autoreload 가 기존 클래스를 갈아끼우지
#  못하고 매 셀 실행마다 아래 오류를 뿜는다:
#     ValueError: __init__() requires a code object with 8 free vars
#  제외해 두면 오류가 사라지고, config.py 를 고쳤을 때만 커널을 재시작하면 된다.
%aimport -src.config

from src.config import CFG

#  config.py 는 dataclass 다. 필드 구조가 바뀌면 %autoreload 가 기존 클래스를
#  갈아끼우지 못하고 «옛 CFG» 가 남는다 (ValueError: ... free vars).
#  그 상태로 진행하면 CFG.send 같은 새 속성이 없어 한참 뒤에야 터진다.
#  여기서 먼저 잡고 커널 재시작을 요구한다.
_need = ('input_audio', 'input_transcript', 'meeting_title', 'meeting_date',
         'accept', 'send', 'output_layout', 'drive_subfolder', 'notion_target')
_missing = [a for a in _need if not hasattr(CFG, a)]
if _missing:
    raise SystemExit(
        'CFG 가 낡았습니다 (없는 속성: ' + ', '.join(_missing) + ')' + chr(10) +
        'config.py 가 바뀌었는데 autoreload 가 dataclass 를 갱신하지 못한 상태입니다.' + chr(10) +
        '-> Jupyter 상단 «Restart Kernel» 후 1번 셀부터 다시 실행하세요.'
    )

#  커널이 어느 파이썬인지 남긴다. 다른 프로젝트 venv 로 돌면 패키지가 갑자기 없을 수 있다.
print(f'kernel   {sys.executable}')
if 'meeting_minutes' not in sys.executable:
    print('         (주의: 이 프로젝트의 .venv 가 아닙니다. 패키지 누락 시 여기부터 의심)')

for k, v in OVERRIDE.items():
    if v:
        setattr(CFG, k, v)
        print(f'[override] {k} = {v}')
CFG.ensure_dirs()

#  뒤 셀들이 쓰는 값. .env 를 정본으로 하고 여기서 이름만 짧게 받는다.
AUDIO      = CFG.input_audio
TRANSCRIPT = CFG.input_transcript
TITLE      = CFG.meeting_title
DATE       = CFG.meeting_date

print(CFG.summary())
print()
if not AUDIO and not TRANSCRIPT:
    print('입력이 비어 있습니다. .env 의 INPUT_AUDIO 또는 INPUT_TRANSCRIPT 를 채우세요.')
elif AUDIO:
    print(f'입력: 오디오 -> STT 를 돌립니다 ({AUDIO})')
else:
    print(f'입력: 녹취록 -> STT 를 건너뜁니다 ({TRANSCRIPT})')

kernel   c:\Users\skswl\Desktop\Github\v02_quiz_builder\.venv\Scripts\python.exe
         (주의: 이 프로젝트의 .venv 가 아닙니다. 패키지 누락 시 여기부터 의심)
입력      audio=-  transcript=-
회의      title=(자동)  date=(미지정)
STT       medium / auto / ko
추출      mode=cli  model=claude-opus-5  API키=없음
검토      accept=(노트북에서 직접)
산출물    C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes  layout=nested  overwrite=False
전송      send=(로컬만)
드라이브   회의록 / subfolder=slug  gdoc=True
          scope=drive.file
노션      (건너뜀)

입력이 비어 있습니다. .env 의 INPUT_AUDIO 또는 INPUT_TRANSCRIPT 를 채우세요.


## 2. STT — 오디오를 텍스트로

**하는 일** — 오디오면 음성 인식을 돌려 녹취록을 만들고, 녹취록이면 그 파일을 그대로 씁니다.
어느 쪽이든 뒤 셀이 쓸 `transcript_path` 와 `source_name` 을 확정합니다.

**녹취록으로 시작하는 경우에도 이 셀은 실행해야 합니다** — 변수가 여기서 할당됩니다.
STT 만 건너뛰는 것이고, 셀을 건너뛰면 다음 셀에서 `NameError` 가 납니다.

**오디오일 때 알아둘 것**

- 이 PC 는 AMD GPU 라 **CPU 로만 돕니다** (faster-whisper 는 NVIDIA CUDA 전용)
- 진행바에 `0.51x` 처럼 배속이 찍힙니다. 1 미만이면 오디오 길이보다 오래 걸립니다
- 느리면 `.env` 의 `WHISPER_MODEL` 을 `medium` → `small` 로 낮추세요
- 같은 이름의 녹취록이 있으면 덮어쓰지 않고 `_v2` 로 저장합니다

In [3]:
#  입력은 .env 의 INPUT_AUDIO / INPUT_TRANSCRIPT 에서 정합니다.
#  둘 다 비어 있으면 여기서 멈춥니다 (예전엔 Path('') -> '.' 가 되어
#   폴더를 읽으려다 PermissionError 가 났습니다).

if not AUDIO and not TRANSCRIPT:
    raise SystemExit(
        '입력이 비어 있습니다.' + chr(10) +
        '  .env 의 INPUT_AUDIO 또는 INPUT_TRANSCRIPT 를 채우고 커널을 재시작하세요.' + chr(10) +
        '  예) INPUT_TRANSCRIPT=data/transcripts/mom_test1.txt'
    )

if AUDIO:
    from src.transcribe import transcribe
    audio_path = Path(AUDIO)
    #  상대경로면 .env 의 오디오 폴더(data/audio) 기준으로 푼다
    if not audio_path.is_absolute() and not audio_path.exists():
        audio_path = CFG.audio_dir / audio_path.name
    if not audio_path.is_file():
        raise SystemExit(f'오디오 파일이 없습니다: {audio_path}')
    transcript_path = transcribe(audio_path)
    source_name = audio_path.name
else:
    transcript_path = Path(TRANSCRIPT)
    if not transcript_path.is_absolute() and not transcript_path.exists():
        transcript_path = CFG.transcript_dir / transcript_path.name
    if not transcript_path.is_file():
        raise SystemExit(
            f'녹취록 파일이 아닙니다: {transcript_path}' + chr(10) +
            '  .env 의 INPUT_TRANSCRIPT 경로를 확인하세요.'
        )
    source_name = transcript_path.name
    print(f'STT 건너뜀 — 기존 녹취록 사용: {transcript_path}')

STT 건너뜀 — 기존 녹취록 사용: .


## 3. 녹취록 확인

**하는 일** — 추출 전에 «어느 파일을 읽었는지, 내용이 멀쩡한지» 눈으로 확인합니다.
파일명·글자수·줄수와 앞 15줄, 뒤 5줄을 보여줍니다.

여기가 깨져 있으면(빈 파일·인코딩 오류·엉뚱한 파일) 뒤 단계가 다 어긋나므로 먼저 봅니다.

`TRANSCRIPT_OVERRIDE` 는 **1·2번을 다시 돌리지 않고 이 셀에서만** 다른 파일을 지정할 때 씁니다.
평소엔 비워두세요.

In [4]:
# ---- 이 셀에서 녹취록을 «직접» 지정하고 싶을 때만 채운다 ----
#  평소엔 비워둡니다. 상시 설정은 .env 의 INPUT_TRANSCRIPT 를 쓰세요.
#  경로는 반드시 r"..." (raw string). r 이 없으면 \U 가 유니코드 이스케이프로
#  해석돼 SyntaxError 가 납니다. 상대경로가 가장 안전합니다.
TRANSCRIPT_OVERRIDE = r''        # 예: r'data/transcripts/mom_test1.txt'

if TRANSCRIPT_OVERRIDE:
    transcript_path = Path(TRANSCRIPT_OVERRIDE)
    if not transcript_path.is_absolute() and not transcript_path.exists():
        transcript_path = CFG.transcript_dir / transcript_path.name
    source_name = transcript_path.name
    print(f'[override] 이 셀에서 지정한 녹취록을 사용합니다: {transcript_path}')

if not transcript_path.is_file():
    raise SystemExit(f'녹취록 파일이 아닙니다: {transcript_path}')

transcript = transcript_path.read_text(encoding='utf-8')
lines = transcript.splitlines()
if not transcript.strip():
    raise SystemExit(f'녹취록이 비어 있습니다: {transcript_path}')

print(f'{transcript_path.name}  |  {len(transcript):,}자 / {len(lines):,}줄')
print('-' * 60)
for l in lines[:15]:
    print(l)
print()
print('... (중략) ...')
print()
for l in lines[-5:]:
    print(l)

PermissionError: [Errno 13] Permission denied: '.'

## 4. 추출 — 녹취록을 회의록 구조로

**하는 일** — `claude -p` 로 Claude Code 를 불러 녹취록을 읽히고, **구조화된 JSON** 을 받습니다.
받은 JSON 을 스키마로 검증하고, 인용이 실제 녹취록에 있는지 대조한 뒤 저장합니다.

```
녹취록 ──▶ claude -p ──▶ JSON ──▶ 스키마 검증 ──▶ 인용 검증 ──▶ draft 저장
```

**추출하는 것** (스키마: `src/schema.py` 의 `Minutes`)

| 항목 | 내용 |
|---|---|
| `topics` | 무슨 얘기를 했나 (시간순) |
| `decisions` | 확정된 것 + 왜 그렇게 정했나 |
| `action_items` | 할 일 + 담당·마감 (없으면 «왜 없는지» 구분) |
| `open_questions` | 결론 안 난 쟁점 |
| `unclear_notes` | STT 가 불확실한 구간 (숫자·날짜·고유명사) |

**만드는 변수** — `minutes`(구조 데이터), `bundle`(메타데이터 포함), `cli_json`(저장 경로)

**몇 분 걸립니다.** API 키는 쓰지 않습니다.

**셀이 스스로 검사하는 것 3가지**

1. **스키마 검증** — 필드·타입이 맞는지 (`Minutes.model_validate`)
2. **인용 검증** — 모든 `quote` 가 우리가 넘긴 녹취록에 실제로 있는지
   → 없으면 중단. «다른 파일을 읽음» 과 «없는 발언을 만듦» 을 동시에 잡습니다
3. **제목·날짜 보정** — 모델이 바꿔놨으면 1번 셀 값으로 되돌립니다

실패하면 이유가 출력됩니다. 품질을 바꾸려면 코드가 아니라 `prompts/extract_system.md` 를 고치세요.

In [6]:
# ============================================================================
#  [대안] CLI 를 API 처럼 쓰는 방법 — 입력 대체를 «구조적으로» 막는다
# ============================================================================
#  왜 필요한가
#    지금 방식은 녹취록 «경로» 를 넘긴다. CLI 는 디스크 접근권이 있어서
#    읽는 김에 폴더를 훑고, 더 «회의다운» 파일이 있으면 바꿔 읽는다.
#    실제로 mom_test1.txt(가족 통화) 대신 옆에 있던 테스트회의.txt 를 읽었다.
#    악의가 아니라 «선의의 판단» 이라 프롬프트로는 완전히 막기 어렵다.
#
#  API 는 이 사고가 불가능하다
#    extract.py 는 녹취록 «내용» 을 프롬프트에 박아 보낸다.
#    모델은 다른 파일이 있는지조차 모른다 -> 고를 수가 없다.
#
#  CLI 를 그 상태로 만드는 방법: 경로가 아니라 내용을 stdin 으로 넣는다
#
#    r = subprocess.run(
#        [CLAUDE, '-p', prompt_without_path, '--append-system-prompt', SYS],
#        input=transcript,                 # <- 파일이 아니라 텍스트를 직접 준다
#        capture_output=True, text=True, encoding='utf-8', errors='replace',
#        env=CHILD_ENV, cwd=str(ROOT), timeout=3600,
#    )
#
#    프롬프트에서 파일 경로 줄을 빼고 이렇게 바꾼다:
#      '표준입력으로 들어온 텍스트가 녹취록이다. 파일을 읽지 않는다.'
#
#  트레이드오프 (그래서 기본값으로 두지 않았다)
#    - 장점: 폴더를 탐색할 이유가 없어져 입력 대체가 원천 차단된다
#    - 단점: 녹취록 전량이 한 번에 컨텍스트로 들어간다. 파일 경로 방식은
#            필요한 만큼만 읽으며 긴 회의를 나눠 처리할 수 있다.
#    - 단점: 규칙·스키마 파일은 여전히 읽어야 하므로 파일 접근을 아예
#            없앨 수는 없다 (읽기 대상이 줄 뿐이다).
#
#  현재 판단: 경로 방식 + 아래 «인용 검증 가드» 로 잡는다.
#             실패하면 즉시 멈추고 이유를 보여주므로 조용히 틀리지 않는다.
#             긴 회의에서 문제가 생기면 그때 stdin 방식으로 옮긴다.
# ============================================================================

import json, os, re, shutil, subprocess
from datetime import datetime
from src.schema import Minutes, MinutesBundle

#  .env 의 EXTRACT_MODE 가 'cli' 여야 이 셀을 쓴다.
#  'api' 로 두면 아래 «참고» 셀의 extract() 방식을 써야 한다.
assert CFG.extract_mode == 'cli', (
    f"EXTRACT_MODE={CFG.extract_mode!r} 입니다. 이 셀은 'cli' 전용입니다. "
    '.env 를 고치거나 아래 API 방식 셀을 쓰세요.'
)
CLAUDE = shutil.which('claude')
assert CLAUDE, 'claude CLI 가 없습니다: npm i -g @anthropic-ai/claude-code'
CHILD_ENV = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}

# 출력 형식은 시스템 프롬프트 층에서 강제한다. 사용자 프롬프트로는 이기지 못한다.
SYS = (
    '너는 JSON 추출 엔드포인트다. 사람에게 보고하지 않는다.'
    ' 최종 응답은 JSON 객체 하나여야 한다.'
    ' 설명·요약·머리말·꼬리말·코드펜스·마크다운을 절대 쓰지 않는다.'
    ' 응답의 첫 글자는 { 이고 마지막 글자는 } 이다. 파일을 만들지 않는다.'
)

task = [
    '녹취록 파일을 읽고, 그 내용에서 회의록을 구조화해 Minutes JSON 을 만들어라.',
    '이 프롬프트에는 녹취록 «본문이 들어 있지 않다». 아래 경로의 파일을 직접 열어서 읽어라.',
    '',
    f'녹취록 파일 (이 파일만 읽는다): {transcript_path}',
    f'회의 제목: {TITLE or "(미지정 - 내용에서 생성)"}',
    f'회의 날짜: {DATE or "(미지정)"}   <- 확정된 사실이다. 상대 날짜(다음 주 월요일 등)는 이 날짜를 기준으로 환산한다.',
    '',
    '읽을 것 (읽기만 한다. 파일을 만들지 않는다):',
    '1. prompts/extract_system.md — 규칙. 그대로 따른다. 새로 만들거나 요약하지 않는다.',
    '2. src/schema.py 의 Minutes — 필드와 허용값. 추측하지 않는다.',
    '3. 위에 지정된 녹취록 «그 파일만».',
    '',
    '입력 고정 (중요):',
    '- 위에 지정된 파일 «하나만» 읽는다. 같은 폴더의 다른 파일(_v2 등)을 읽거나 대체하지 않는다.',
    '- 어느 파일을 쓸지 사용자에게 되묻지 않는다. 경로는 이미 확정돼 있다.',
    '- 내용이 회의처럼 보이지 않아도(잡담·통화 등) 그 파일을 그대로 처리한다.',
    '  더 «회의다운» 파일을 찾아 바꾸지 않는다. 판단은 사람이 한다.',
    '- 추출할 결정·액션이 없으면 빈 배열로 두고 topics 만 채운다. 다른 파일로 갈아타지 않는다.',
    # 규칙을 여기에 다시 적지 않는다. 두 곳에 있으면 갈라진다 (정본은 prompts/extract_system.md).
]
prompt = chr(10).join(task)

print('추출 중... (녹취록 길이에 따라 몇 분 걸립니다)')
r = subprocess.run(
    [CLAUDE, '-p', prompt, '--append-system-prompt', SYS],
    capture_output=True, text=True, encoding='utf-8', errors='replace',
    env=CHILD_ENV, cwd=str(ROOT), timeout=3600,
)
raw_out = (r.stdout or '').strip()
if r.returncode != 0:
    print(f'exit {r.returncode}')
    print((r.stderr or '')[-800:])


def extract_json(text):
    """응답에서 JSON 본문만 꺼낸다. 코드펜스나 앞뒤 설명이 붙어도 견딘다."""
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.S)
    if m:
        return m.group(1)
    i, j = text.find('{'), text.rfind('}')
    return text[i:j + 1] if (i != -1 and j > i) else None


payload = extract_json(raw_out)
assert payload, ('응답에서 JSON 을 찾지 못했습니다. 받은 내용:' + chr(10) + raw_out[:1200])

data = json.loads(payload)
minutes = Minutes.model_validate(data['minutes'] if 'minutes' in data else data)


def verify_quotes(minutes, transcript):
    """모든 quote 가 «우리가 넘긴» 녹취록에 실제로 있는지 확인한다.

    이 가드는 두 가지를 동시에 잡는다.
      1. 에이전트가 다른 녹취록 파일을 읽어버린 경우 (실제로 발생했다)
      2. 없는 발언을 만들어낸 경우
    공백만 무시하고 부분문자열로 대조한다.
    """
    flat = re.sub(r'\s+', '', transcript)
    bad = []
    for kind, items in (('D', minutes.decisions), ('A', minutes.action_items)):
        for n, it in enumerate(items, 1):
            if re.sub(r'\s+', '', it.quote or '') not in flat:
                bad.append((f'{kind}{n}', it.quote))
    return bad


bad_quotes = verify_quotes(minutes, transcript)
if bad_quotes:
    print('!! 인용 검증 실패 — 녹취록에 없는 발언이 있습니다')
    print(f'   대상 녹취록: {transcript_path}')
    for label, q in bad_quotes:
        print(f'   {label}: {(q or "")[:70]}')
    raise AssertionError(
        '인용이 녹취록과 일치하지 않습니다. 다른 파일을 읽었거나 발언을 만들어낸 것입니다. '
        'data/transcripts 에 다른 파일이 섞여 있는지 확인하고 다시 실행하세요.'
    )
print(f'인용 검증 통과 ({len(minutes.decisions) + len(minutes.action_items)}건 전부 녹취록에 존재)')

# 논의 연결 확인. 인용과 달리 «막지 않고 알린다» — 논의가 빠진 것인지
# 애초에 잡담이라 항목이 아닌 것인지는 사람이 판단할 문제다.
from src.review import orphan_items

orphans = orphan_items(minutes)
if orphans:
    print()
    print(f'!! 논의 연결 없음 {len(orphans)}건 — 논의 내용에서 근거를 되짚을 수 없습니다')
    for label, text in orphans:
        print(f'   {label}: {text}')
    print('   → 논의가 빠졌으면 3번 셀 재실행, 잡담이면 5번에서 빼고 고르세요')
else:
    print('논의 연결 통과 (모든 항목이 논의 내용에 연결됨)')

# 우리가 아는 값은 모델 결과를 신뢰하지 않는다 (제목 변경·날짜 누락을 실제로 확인).
fix = {}
if TITLE and minutes.title != TITLE:
    print(f'제목 보정: {minutes.title!r} -> {TITLE!r}')
    fix['title'] = TITLE
if DATE and minutes.date != DATE:
    print(f'날짜 보정: {minutes.date!r} -> {DATE!r}')
    fix['date'] = DATE
if fix:
    minutes = minutes.model_copy(update=fix)

bundle = MinutesBundle(
    minutes=minutes,
    source_audio=source_name,
    transcript_chars=len(transcript),
    model='claude-code(cli)',
    generated_at=datetime.now().strftime('%Y-%m-%d %H:%M'),
)

# 셀이 직접 저장한다 (Claude 에게 쓰기 권한을 요구하지 않는다)
draft_dir = CFG.output_dir / 'draft'
draft_dir.mkdir(parents=True, exist_ok=True)
stem = ((DATE + '_') if DATE else '') + (TITLE or transcript_path.stem)
cli_json = draft_dir / (stem + '.cli.json')
cli_json.write_text(json.dumps(bundle.model_dump(), ensure_ascii=False, indent=2), encoding='utf-8')

print()
print(f'검증 통과 · 결정 {len(minutes.decisions)} · 액션 {len(minutes.action_items)} '
      f'· 미결 {len(minutes.open_questions)}')
print(f'저장: {cli_json}')

추출 중... (녹취록 길이에 따라 몇 분 걸립니다)
인용 검증 통과 (3건 전부 녹취록에 존재)
논의 연결 통과 (모든 항목이 논의 내용에 연결됨)

검증 통과 · 결정 1 · 액션 2 · 미결 1
저장: C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes\draft\2026-08-25_킥오프.cli.json


<details><summary>참고 — API 키로 추출하는 경우 (지금은 안 씀)</summary>

`.env` 에 `ANTHROPIC_API_KEY` 가 있으면 4번 셀 대신 아래를 쓸 수 있습니다.
긴 회의를 자동으로 분할 추출·병합해주는 것이 장점입니다.

```python
from datetime import datetime
from src.extract import extract
from src.schema import MinutesBundle

minutes = extract(transcript, title=TITLE or None, date=DATE or None)
bundle = MinutesBundle(minutes=minutes, source_audio=source_name,
                      transcript_chars=len(transcript), model=CFG.model,
                      generated_at=datetime.now().strftime('%Y-%m-%d %H:%M'))
```

단 CLI 경로에 있는 **인용 검증 가드가 없습니다.** 쓰려면 그 부분을 옮겨야 합니다.

</details>

## 5. 검토 — 무엇을 반영할지 고르기

**하는 일** — 추출된 항목에 번호를 붙여 보여줍니다. 문서는 아직 만들지 않습니다.

```
D1 D2 …  결정사항      A1 A2 …  액션아이템      Q1 Q2 …  미결 사항
```

**회의에서 나온 말이 전부 결정은 아닙니다.** 어느 것이 결정인지는 참석한 사람만 알기 때문에
여기서 한 번 멈춥니다.

**같이 나오는 경고**

| 표시 | 뜻 | 다음 행동 |
|---|---|---|
| 회의에서 안 정해짐 | 녹취를 다 봤지만 회의에서 안 정함 | **참석자에게 묻는다** |
| 녹취 불확실 | 녹취가 깨져 확인 못 함 | **원본 오디오를 다시 듣는다** |
| 논의 연결 없음 | 논의 내용에서 근거를 되짚을 수 없음 | 논의가 빠졌나 / 잡담인가 확인 |

In [7]:
from src.review import render_review, blank_report

print(render_review(minutes))

r = blank_report(minutes)
if r["not_stated"]:
    print()
    print(f"! 회의에서 담당/마감을 정하지 않은 액션 {r['not_stated']}건 — 참석자에게 확인")
if r["unclear"]:
    print(f"! 녹취가 불확실해 확인 못 한 액션 {r['unclear']}건 — 원본 오디오 재확인")
if r["notes"]:
    print(f"! 녹취 불확실 구간 {r['notes']}건 — 확정 전 확인 필요")

  킥오프   2026-08-25
  업무 회의가 아닌 가족 안부 통화 — 엄마 마사지는 받지 않기로 하고, 대신 포도를 챙기는 쪽으로 정리

[결정사항]
  D1. 엄마 마사지는 받지 않기로 함
      논의: 엄마 마사지 여부
      근거: "아니 안 받아도 돼 지금 요즘에 일 안할게 그래도 태어나서 엄마가" ·00:01:59

[액션아이템]
  A1. 엄마에게 포도를 사서 챙겨 드리기
      논의: 엄마 마사지 여부
      담당 <녹취 불확실 · 오디오 재확인> / 마감 <회의에서 안 정해짐> / medium
  A2. 포도·복숭아 등 제철 과일을 사서 씻어 깎아 챙겨 먹기
      논의: 제철 과일 챙겨 먹기
      담당 <녹취 불확실 · 오디오 재확인> / 마감 <회의에서 안 정해짐> / low

[미결 사항]
  Q1. 누나가 알아보던 마사지가 이미 예약된 상태인지, 취소를 전달해야 하는지 확인되지 않음
      논의: 엄마 마사지 여부

[녹취 불확실 — 확정 전 확인 필요]
  · 전 구간에 화자 라벨이 없다 — 통화 양쪽 발언과 옆사람 발언이 섞여 있어 발언 주체를 특정할 수 없다. 액션 담당자를 모두 unclear 로 둔 이유이며, 담당자 확정에는 원본 오디오 재확인이 필요하다
  · "차라리 그놈이라 포도 먹고싶다"(00:02:13) — "그 돈으로"의 오인식으로 추정 (구버전 전사: "난 차라리 그 놈으로 포도 먹고 싶다"). 액션아이템 A1 의 근거 문장이므로 우선 재확인 대상
  · "지금 요즘에 일 안할게 그래도 태어나서 엄마가"(00:01:59) — 뒷부분이 깨짐. 구버전 전사는 "일 안 한 게 그래도 더 낫어 엄마가". 결정사항 D1 의 사유 구간이므로 재확인 필요
  · "바테도 받아도 돼?"(00:01:56) · "엄마 바테나가서 일하면"(00:02:06) — "밭에 (나가서)"의 오인식으로 추정 (구버전: "엄마 밭에 가서 일해면")
  · "엄마가 또 배로그거든"(00:02:06) — 의미 불확실 (구버

## 6. 확정 — 고른 것만 문서로

**하는 일** — `ACCEPT` 에 적은 항목만 남겨 **md · html · json** 세 파일을 만듭니다.

```
ACCEPT = "all"           전부 반영
ACCEPT = "D1,A1,A3"      고른 것만
```

**만드는 변수** — `out` (세 파일 경로). 뒤의 전송 단계가 이걸 씁니다.

| 파일 | 용도 |
|---|---|
| `.md` | 사람이 읽는 회의록 (노션·슬랙 붙여넣기) |
| `.html` | 뷰어 (액션 표 + 근거 인용) |
| `.json` | 구조 데이터 (다음 자동화 입력) |

고른 항목을 바꾸려면 **이 셀만** 다시 실행하면 됩니다. 같은 제목·날짜면 `_v2` 로 저장됩니다.

In [8]:
ACCEPT = CFG.accept or "all"        # .env 의 ACCEPT. 비우면 all

from src.review import parse_accept, apply_selection
from src.render import render

picked, unknown = parse_accept(ACCEPT, minutes)
assert not unknown, f"알 수 없는 라벨: {unknown}"
assert picked, "선택된 항목이 없습니다"

final = bundle.model_copy(update={"minutes": apply_selection(minutes, picked)})
out = render(final)
print(f"반영 {len(picked)}개: {', '.join(picked)}")
print(f"md   {out.md}")
print(f"html {out.html}")
print(f"json {out.json}")

[render] 같은 이름이 이미 있어 2026-08-25_킥오프_v3 로 저장합니다 (덮어쓰지 않음)
[render] C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes\2026-08-25\킥오프 에 2026-08-25_킥오프_v3.md / 2026-08-25_킥오프_v3.html / 2026-08-25_킥오프_v3.json
반영 4개: D1, A1, A2, Q1
md   C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes\2026-08-25\킥오프\2026-08-25_킥오프_v3.md
html C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes\2026-08-25\킥오프\2026-08-25_킥오프_v3.html
json C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes\2026-08-25\킥오프\2026-08-25_킥오프_v3.json


### 6-1. 결과 미리보기

**하는 일** — 방금 만든 회의록을 노트북 안에서 바로 보여줍니다. 파일을 열지 않아도 확인됩니다.

`OPEN_HTML = True` 로 두면 HTML 뷰어를 **기본 브라우저로** 엽니다.
노트북 안 `IFrame` 은 쓰지 않습니다 — 산출물이 노트북 폴더 밖(`data/minutes/<날짜>/<제목>/`)에 있어
상대경로가 잡히지 않고 빈 프레임이 마크다운 출력을 덮습니다.

In [ ]:
OPEN_HTML = CFG.open_html        # .env 의 OPEN_HTML

from IPython.display import Markdown, display

text = out.md.read_text(encoding="utf-8")
print(f'{out.md.name}  |  {len(text):,}자')
print(f'{out.md.parent}')
print('-' * 60)
display(Markdown(text))

if OPEN_HTML:
    import subprocess
    #  노트북 폴더 밖이라 IFrame 상대경로가 안 잡힌다. 브라우저로 연다.
    subprocess.run(['cmd', '/c', 'start', '', str(out.html)])
    print(f'브라우저로 열었습니다: {out.html.name}')

## 7. 구글 드라이브 전송

**하는 일** — 회의록 파일을 드라이브로 보냅니다. **어느 계정 · 어느 폴더** 인지 먼저 확인하고,
확인이 통과했을 때만 전송합니다.

```
7-1  확인   계정 · 저장 경로 · 준비 상태     ← 파일을 옮기지 않는다
7-2  전송   7-1 통과 시에만
```

### 경로는 어디서 바꾸나 — 전부 [`.env`](.env)

| 바꿀 것 | `.env` 키 | 예 |
|---|---|---|
| 전송 방식 | `SEND` | `api` / `sync` / `manual` / (비움) |
| 최상위 폴더 이름 | `DRIVE_FOLDER_NAME` | `회의록` |
| 회의별 하위 폴더 | `DRIVE_SUBFOLDER` | `slug` / `month` / `none` |
| 특정 폴더 안에 넣기 | `DRIVE_PARENT_ID` | 폴더 URL 끝의 ID |
| Docs 변환본 함께 | `DRIVE_AS_GDOC` | `true` / `false` |
| 본인 계정 확인용 | `MY_DRIVE_EMAIL` | `you@gmail.com` |
| 동기화 폴더(sync) | `SYNC_DIR` | `G:\My Drive\회의록` |

> `.env` 를 고쳤으면 **커널 재시작** 후 1번 셀부터 다시 실행해야 반영됩니다.

### 7-1. 확인 (전송 안 함)

«어디에 올라갈지» 를 실제 경로로 보여줍니다. 눈으로 확인하고 7-2 로 넘어가세요.

In [ ]:
# ===== 7-1. 확인 (파일을 옮기지 않는다) =====
#  경로 설정은 모두 .env 에 있습니다:
#    SEND / DRIVE_FOLDER_NAME / DRIVE_SUBFOLDER / DRIVE_PARENT_ID / DRIVE_AS_GDOC
#    MY_DRIVE_EMAIL (본인 계정 확인용) / SYNC_DIR (sync 방식)
#  1번 셀 없이 이 셀만 돌릴 때를 위해 CFG 를 확보한다.
try:
    CFG
except NameError:
    import sys
    from pathlib import Path
    sys.path.insert(0, str(Path.cwd()))
    from src.config import CFG
    print('[자체 로드] CFG 를 직접 가져왔습니다 (1번 셀 미실행)')

import json as _json

OK_API = OK_SYNC = False
SEND_TARGET = None

print(f'SEND = {CFG.send!r}   (.env 의 SEND)')
print()

# ── api 방식 ──────────────────────────────────────────────
print('[api] Drive API 업로드')
cred, tok = CFG.drive_credentials, CFG.drive_token
if not cred.exists():
    print(f'  X  credentials.json 없음 -> {cred}')
    print('     Cloud Console 에서 «데스크톱 앱» OAuth 클라이언트를 만들어 저장하세요.')
else:
    try:
        d = _json.loads(cred.read_text(encoding='utf-8'))
        kind = next(iter(d))
        if kind != 'installed':
            print(f'  X  유형 {kind!r} — «데스크톱 앱» 으로 다시 만드세요.')
        else:
            OK_API = True
            print(f'  O  credentials.json 정상 (project={d[kind].get("project_id")})')
    except Exception as e:
        print(f'  X  읽기 실패: {e}')

if OK_API:
    #  실제 업로드될 경로를 미리 조립해 보여준다 (drive.py 와 같은 규칙)
    _sub = CFG.subfolder_for(out.slug) if 'out' in dir() else '<회의별 폴더>'
    _top = '내 드라이브' if not CFG.drive_parent_id else f'(부모 {CFG.drive_parent_id})'
    _path = ' / '.join([_top, CFG.drive_folder_name] + ([_sub] if _sub else []))
    print(f'     올릴 위치 : {_path}')
    print(f'     규칙      : DRIVE_SUBFOLDER={CFG.drive_subfolder}  DRIVE_AS_GDOC={CFG.drive_as_gdoc}')
    print(f'     스코프    : {CFG.drive_scope.rsplit("/", 1)[-1]}')
    if tok.exists():
        #  토큰이 있으면 «실제 계정» 을 물어본다. 여기서 계정이 틀리면 지금 멈춰야 한다.
        try:
            from src.drive import _account_email, _service
            print(f'     인증 계정 : {_account_email(_service())}')
        except Exception as e:
            print(f'     인증 계정 확인 실패: {str(e)[:120]}')
    else:
        print('     인증 계정 : (첫 실행 시 브라우저에서 선택 — 본인 계정으로 로그인)')

# ── sync 방식 ─────────────────────────────────────────────
print()
print('[sync] Drive for desktop 동기화 폴더')
try:
    from src.drive_accounts import confirm_target, list_accounts
    for a in list_accounts():
        print(f'  - {a.label}' + (' *현재활성' if a.is_current else ''))
    if CFG.my_drive_email:
        try:
            SEND_TARGET = confirm_target(CFG.my_drive_email, CFG.sync_dir or None)
            OK_SYNC = True
            print(f'  O  올릴 위치: {SEND_TARGET}')
        except SystemExit as e:
            print(f'  X  {str(e).splitlines()[0]}')
    else:
        print('  -  MY_DRIVE_EMAIL 이 비어 확인 생략 (.env 에서 지정)')
except Exception as e:
    print(f'  X  {e}')

print()
print('쓸 수 있는 방식:', ', '.join(
    (['api'] if OK_API else []) + (['sync'] if OK_SYNC else []) + ['manual', '(비움)']))

### 7-2. 전송

| `SEND` | 동작 | 필요한 것 |
|---|---|---|
| `"api"` | Drive API 자동 업로드 + 공유 링크 | `credentials.json` |
| `"sync"` | 동기화 폴더로 복사 | Drive 앱에 계정 추가 + `SYNC_DIR` |
| `"manual"` | 탐색기 + drive.google.com 열기 | 없음 |
| `""` | 로컬에만 | 없음 |

6-1 이 «쓸 수 있는 방식» 을 알려줍니다. 준비 안 된 방식을 고르면 그 이유를 출력하고 멈춥니다.

In [15]:
# ===== 7-2. 전송 =====
SEND = CFG.send        # .env 의 SEND ('api'|'sync'|'manual'|'')

if SEND == "api":
    assert OK_API, '7-1 에서 credentials.json 확인이 실패했습니다. 위 셀 출력을 보세요.'
    from src.drive import upload_minutes
    #  403 access_denied 가 나오면 원인은 «테스트 사용자 미등록» 이다.
    #  credentials.json 이 있어도 별개 설정이라 따로 해야 한다.
    #    Console -> Google 인증 플랫폼 -> 대상 -> 테스트 사용자 -> + Add users
    #  6-1 점검으로는 잡을 수 없다 (Google 서버 쪽 설정이라 로컬에서 조회 불가).
    if not CFG.drive_token.exists():
        print('첫 실행입니다 — 브라우저가 열립니다. 본인 계정으로 로그인·승인하세요.')
        print('(«확인되지 않은 앱» 경고가 나오면 고급 -> 계속 진행. 본인이 만든 앱입니다)')
    try:
        res = upload_minutes(out.md, out.html, out.json, subfolder=CFG.subfolder_for(out.slug))
        print(res.report())          # 계정 · 저장 위치 · 폴더 링크 · 파일 링크
        print()
        print('드라이브에서 열기:', res.folder_link)
    except Exception as e:
        msg = str(e)
        print(f'업로드 실패: {msg[:200]}')
        if 'access_denied' in msg or '403' in msg:
            print()
            print('원인: 테스트 사용자로 등록되지 않은 계정입니다.')
            print('  Console -> Google 인증 플랫폼 -> 대상 -> 테스트 사용자 -> + Add users')
            print('  본인 이메일을 추가한 뒤 이 셀을 다시 실행하세요.')
        elif 'invalid_grant' in msg or 'expired' in msg:
            print()
            print('원인: token.json 이 만료/무효입니다. 파일을 지우고 다시 실행하세요.')
            print(f'  {CFG.drive_token}')

elif SEND == "sync":
    assert OK_SYNC, '7-1 계정 확인을 통과하지 못했습니다.'
    from src.sync import sync_files
    print(f'대상: {SEND_TARGET}')
    for s in sync_files([out.md, out.html, out.json], subfolder=CFG.subfolder_for(out.slug)):
        print(f'  {s.dst.name}')
    print('Drive 가 백그라운드로 업로드합니다.')

elif SEND == "manual":
    import subprocess
    print(f'탐색기: {out.md.parent}')
    print('브라우저: drive.google.com  — 파일 3개를 드래그하세요')
    subprocess.run(['explorer', '/select,', str(out.md)])
    subprocess.run(['cmd', '/c', 'start', '', 'https://drive.google.com/drive/my-drive'])

else:
    print('전송 건너뜀 — 로컬에만 저장했습니다.')
    for p in (out.md, out.html, out.json):
        print(f'  {p}')

Failed to reload module 'src.config' from file 'c:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\src\config.py'
Traceback (most recent call last):
  File "c:\Users\skswl\Desktop\Github\v02_quiz_builder\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\skswl\Desktop\Github\v02_quiz_builder\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 625, in superreload
    update_generic(old_obj, new_obj)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "c:\Users\skswl\Desktop\Github\v02_quiz_builder\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 451, in update_generic
    update(a, b)
    ~~~~~~^^^^^^
  File "c:\Users\skswl\Desktop\Github\v02_quiz_builder\.venv\Lib\site-packages\IPython\extensions\autoreload.py", line 403, in update_class
    if update_generic(old_obj, new_obj):
       ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^

AttributeError: 'Config' object has no attribute 'send'

## 8. 노션에 올리기

**하는 일** — `claude -p` 로 Claude Code 를 불러 **노션 커넥터로** 페이지를 만듭니다.
노션 API 토큰을 쓰지 않습니다. 대화형 세션을 열어둘 필요도 없습니다 (셀이 CLI 를 띄우고 끝냄).

```
8-1  확인   계정 · 워크스페이스 · 올릴 상위 페이지     ← 올리지 않는다
8-2  올리기 8-1 통과 시에만
```

### 경로는 어디서 바꾸나

| 바꿀 것 | 어디 | 예 |
|---|---|---|
| 올릴 상위 페이지 | [`.env`](.env) 의 `NOTION_TARGET` | 페이지 이름 또는 URL |
| 연결 계정·워크스페이스 | claude.ai → 설정 → 커넥터 | — |

`NOTION_TARGET` 이 비어 있으면 노션 단계를 건너뜁니다.
**없는 페이지 이름을 적으면 8-1 이 «찾지 못했다» 고 알려줍니다** — 그때 노션에서 페이지를 만들거나
URL 을 그대로 넣으세요.

### 8-1. 확인 (올리지 않음)

노션 도구가 붙는지, **어느 계정·워크스페이스**인지, **올릴 상위 페이지가 실제로 있는지** 확인합니다.
계정을 안 보고 올리면 엉뚱한 워크스페이스에 회의록이 생깁니다.

In [1]:
# ===== 8-1. 확인 =====
#  올릴 위치는 .env 의 NOTION_TARGET 에서 바꿉니다 (페이지 이름 또는 URL).
#  연결 계정·워크스페이스는 claude.ai → 설정 → 커넥터에서 바꿉니다.
#  1번 셀 없이 이 셀만 돌릴 때를 위해 CFG 를 확보한다.
try:
    CFG
except NameError:
    import sys
    from pathlib import Path
    sys.path.insert(0, str(Path.cwd()))
    from src.config import CFG
    print('[자체 로드] CFG 를 직접 가져왔습니다 (1번 셀 미실행)')

import os, re, shutil, subprocess

CLAUDE = shutil.which('claude')
CHILD_ENV = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
print('claude CLI :', CLAUDE or '없음 - npm i -g @anthropic-ai/claude-code')
print('NOTION_TARGET :', CFG.notion_target or '(비어 있음 — .env 에서 지정)')
print()

NOTION_OK = False
NOTION_INFO = {}

if not CFG.notion_target:
    print('NOTION_TARGET 이 비어 있어 노션 단계를 건너뜁니다.')
elif CLAUDE:
    #  도구 유무 + 계정 + 워크스페이스 + «대상 페이지 실존» 을 한 번에 확인한다.
    probe = chr(10).join([
        '노션 커넥터 상태를 확인해라. 아래 네 줄«만» 출력하고 다른 말은 하지 마라.',
        'TOOLS: <notion 도구 개수 숫자만, 없으면 0>',
        'ACCOUNT: <내 노션 이름과 이메일. 확인 불가면 UNKNOWN>',
        'WORKSPACE: <워크스페이스 이름. 개인 워크스페이스면 personal>',
        f'TARGET: <"{CFG.notion_target}" 로 검색해 찾은 페이지의 제목과 URL. 없으면 NOTFOUND>',
        '',
        '계정은 notion-get-users 에 user_id="self" 로, 대상은 notion-search 로 확인해라.',
        '페이지를 만들지 마라. 확인만 한다.',
    ])
    try:
        r = subprocess.run([CLAUDE, '-p', probe], capture_output=True, text=True,
                           encoding='utf-8', errors='replace', env=CHILD_ENV, timeout=300)
        raw = ((r.stdout or '') + (r.stderr or '')).strip()
        for key in ('TOOLS', 'ACCOUNT', 'WORKSPACE', 'TARGET'):
            m = re.search(key + r':\s*(.+)', raw)
            NOTION_INFO[key] = m.group(1).strip() if m else ''
        n = int(re.sub(r'[^0-9]', '', NOTION_INFO.get('TOOLS', '0')) or 0)
        acct = NOTION_INFO.get('ACCOUNT', '')
        tgt = NOTION_INFO.get('TARGET', '')
        print('노션 도구   :', f'{n}개' if n else '없음')
        print('연결 계정   :', acct or '(확인 안 됨)')
        print('워크스페이스 :', NOTION_INFO.get('WORKSPACE') or '(확인 안 됨)')
        print('올릴 상위   :', tgt or '(확인 안 됨)')
        NOTION_OK = (n > 0 and 'UNKNOWN' not in acct.upper()
                     and 'NOTFOUND' not in tgt.upper() and bool(tgt))
        if not NOTION_OK:
            print()
            if n == 0:
                print('  -> claude.ai 설정 > 커넥터에서 Notion 을 인증하세요.')
            elif 'NOTFOUND' in tgt.upper():
                print(f'  -> 노션에 "{CFG.notion_target}" 페이지가 없습니다.')
                print('     노션에서 페이지를 만들거나, .env 의 NOTION_TARGET 에 페이지 URL 을 넣으세요.')
    except subprocess.TimeoutExpired:
        print('시간 초과 — 권한 확인 대기 중일 수 있습니다')

print()
print('노션 업로드 가능:', NOTION_OK)

claude CLI : C:\Users\skswl\AppData\Roaming\npm\claude.CMD


NameError: name 'CFG' is not defined

### 8-2. 올리기

**하는 일** — 회의록 파일 경로를 `claude -p` 에 넘겨 노션 페이지를 만들게 합니다.
회의 내용이 명령줄에 노출되지 않도록 **경로만** 전달하고, Claude 가 파일을 직접 읽습니다.

In [ ]:
NOTION_TARGET = CFG.notion_target        # .env 의 NOTION_TARGET

task = [
    f'다음 회의록 파일을 읽고 노션에 페이지로 올려줘.',
    f'파일: {out.md}',
    f'구조 데이터(참고용): {out.json}',
    f'올릴 위치: 노션의 {NOTION_TARGET}',
    '',
    '규칙:',
    '- 파일에 있는 내용만 쓴다. 요약하거나 새로 만들지 않는다.',
    '- 담당/마감이 <회의에서 안 정해짐> 또는 <녹취 불확실>이면 그 표기를 그대로 유지한다.',
    '- 액션아이템은 체크박스(to-do) 블록으로 만든다.',
    '- 결정사항은 근거 인용을 함께 남긴다.',
    '- 다 끝나면 만든 페이지 URL 한 줄만 마지막에 출력한다.',
]
prompt = chr(10).join(task)

if not NOTION_OK:
    print('헤드리스에서 노션 도구가 없습니다. 채팅창에 아래를 붙여넣으세요:')
    print()
    print(prompt)
else:
    print('노션 업로드 중... (몇 분 걸릴 수 있습니다)')
    r = subprocess.run([CLAUDE, '-p', prompt], capture_output=True, text=True,
                       encoding='utf-8', errors='replace', env=CHILD_ENV, timeout=900)
    res = ((r.stdout or '') + (r.stderr or '')).strip()
    print(res[-2000:])
    if r.returncode != 0:
        print()
        print(f'실패 (exit {r.returncode})')
        print('권한 확인에서 멈춘 경우: 아래처럼 도구를 미리 허용하고 다시 시도하세요.')
        print("  subprocess.run([CLAUDE, '-p', prompt, '--allowedTools', 'mcp__notion'])")

---

## 다시 돌릴 때

| 하고 싶은 것 | 실행할 셀 |
|---|---|
| 고르는 항목만 바꾸기 | 5번만 (`ACCEPT` 수정) |
| 추출 품질이 아쉬움 | `prompts/extract_system.md` 고치고 3번부터 |
| 다른 회의 | 1번부터 (`AUDIO`/`TITLE`/`DATE` 수정) |
| STT 다시 | 2번부터 (`.env` 의 `WHISPER_MODEL` 조정) |

**커밋 전에 `Kernel → Restart & Clear All Outputs`** — 출력에 회의 내용이 남아 있습니다.